# Previsão de Aprovação de Empréstimo com Árvore de Decisão

**Tarefa:** classificação binária — prever se um pedido de empréstimo será **aprovado** (`Approved`) ou **rejeitado** (`Rejected`).

**Dataset:** [Loan Approval Prediction Dataset](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset) (Kaggle).

**Algoritmo:** `DecisionTreeClassifier` (scikit-learn).

Etapas:
1. Download e carregamento dos dados
2. Análise exploratória (EDA)
3. Pré-processamento
4. Divisão treino/teste
5. Modelo base (árvore sem restrições)
6. Ajuste de hiperparâmetros com validação cruzada (`GridSearchCV`)
7. Avaliação no conjunto de teste
8. Interpretação: visualização da árvore, regras e importância das variáveis

## 1. Importação das bibliotecas

In [ ]:
# !pip install kagglehub scikit-learn pandas matplotlib seaborn  # descomente se necessário

import glob
import os

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Download e carregamento do dataset

In [ ]:
# Download latest version
path = kagglehub.dataset_download("architsharma01/loan-approval-prediction-dataset")

print("Path to dataset files:", path)
print("Arquivos:", os.listdir(path))

In [ ]:
csv_path = glob.glob(os.path.join(path, "*.csv"))[0]
df = pd.read_csv(csv_path)

print("Formato:", df.shape)
df.head()

## 3. Análise exploratória (EDA)

Os nomes das colunas e os valores categóricos deste dataset vêm com **espaços em branco no início** (ex.: `' loan_status'`, `' Approved'`). Vamos limpá-los antes de qualquer análise.

In [ ]:
# Remove espaços dos nomes das colunas e dos valores de texto
df.columns = df.columns.str.strip()
for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].str.strip()

df.info()

In [ ]:
df.describe().T

In [ ]:
print("Valores ausentes por coluna:")
print(df.isnull().sum())
print("\nLinhas duplicadas:", df.duplicated().sum())

### 3.1 Distribuição da variável alvo

In [ ]:
print(df["loan_status"].value_counts())
print()
print(df["loan_status"].value_counts(normalize=True).round(3))

ax = sns.countplot(data=df, x="loan_status", hue="loan_status", palette="Set2", legend=False)
ax.set_title("Distribuição de loan_status")
plt.show()

### 3.2 Variáveis categóricas vs. alvo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["education", "self_employed"]):
    sns.countplot(data=df, x=col, hue="loan_status", palette="Set2", ax=ax)
    ax.set_title(f"{col} x loan_status")
plt.tight_layout()
plt.show()

### 3.3 Variáveis numéricas vs. alvo

In [ ]:
num_cols = [
    "no_of_dependents", "income_annum", "loan_amount", "loan_term", "cibil_score",
    "residential_assets_value", "commercial_assets_value",
    "luxury_assets_value", "bank_asset_value",
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x="loan_status", y=col, hue="loan_status", palette="Set2", legend=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

O `cibil_score` (score de crédito) separa muito bem as duas classes — é de se esperar que ele apareça no topo da árvore.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação entre variáveis numéricas")
plt.show()

## 4. Pré-processamento

- Remover `loan_id` (identificador, sem valor preditivo).
- Codificar as variáveis categóricas binárias em 0/1.
- Codificar o alvo: `Approved = 1`, `Rejected = 0`.

Árvores de decisão **não precisam** de normalização/padronização das variáveis numéricas, pois as divisões são feitas por limiares em cada variável individualmente.

In [ ]:
data = df.drop(columns=["loan_id"]).copy()

data["education"] = data["education"].map({"Graduate": 1, "Not Graduate": 0})
data["self_employed"] = data["self_employed"].map({"Yes": 1, "No": 0})
data["loan_status"] = data["loan_status"].map({"Approved": 1, "Rejected": 0})

assert data.isnull().sum().sum() == 0, "Há valores não mapeados!"

X = data.drop(columns=["loan_status"])
y = data["loan_status"]

print("X:", X.shape, "| y:", y.shape)
X.head()

## 5. Divisão treino/teste

80% treino / 20% teste, com estratificação para manter a proporção das classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporção de aprovados — treino: %.3f | teste: %.3f" % (y_train.mean(), y_test.mean()))

## 6. Modelo base: árvore sem restrições

Uma árvore sem limite de profundidade tende a decorar o conjunto de treino (*overfitting*). Usamos este modelo como referência.

In [ ]:
def avaliar(modelo, X_tr, y_tr, X_te, y_te, nome="Modelo"):
    y_pred_tr = modelo.predict(X_tr)
    y_pred_te = modelo.predict(X_te)
    y_proba_te = modelo.predict_proba(X_te)[:, 1]
    return pd.Series({
        "acurácia_treino": accuracy_score(y_tr, y_pred_tr),
        "acurácia_teste": accuracy_score(y_te, y_pred_te),
        "precisão": precision_score(y_te, y_pred_te),
        "recall": recall_score(y_te, y_pred_te),
        "f1": f1_score(y_te, y_pred_te),
        "roc_auc": roc_auc_score(y_te, y_proba_te),
        "profundidade": modelo.get_depth(),
        "nº_folhas": modelo.get_n_leaves(),
    }, name=nome)


arvore_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
arvore_base.fit(X_train, y_train)

res_base = avaliar(arvore_base, X_train, y_train, X_test, y_test, "Árvore base")
res_base.to_frame().round(4)

## 7. Ajuste de hiperparâmetros (GridSearchCV)

Buscamos a melhor combinação de hiperparâmetros com **validação cruzada estratificada de 5 folds**, otimizando o **F1-score**:

| Hiperparâmetro | Função |
|---|---|
| `criterion` | Medida de impureza (`gini` ou `entropy`) |
| `max_depth` | Profundidade máxima da árvore |
| `min_samples_split` | Mínimo de amostras para dividir um nó |
| `min_samples_leaf` | Mínimo de amostras em uma folha |
| `class_weight` | Compensa o desbalanceamento das classes |

In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Melhores hiperparâmetros:", grid.best_params_)
print("Melhor F1 (validação cruzada): %.4f" % grid.best_score_)

In [ ]:
# Top 10 combinações
resultados_cv = pd.DataFrame(grid.cv_results_)
resultados_cv[["params", "mean_test_score", "std_test_score", "rank_test_score"]] \
    .sort_values("rank_test_score").head(10)

### 7.1 Efeito da profundidade (curva de complexidade)

In [ ]:
profundidades = range(1, 21)
acc_treino, acc_val = [], []

for d in profundidades:
    m = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    m.fit(X_train, y_train)
    acc_treino.append(accuracy_score(y_train, m.predict(X_train)))
    acc_val.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(profundidades, acc_treino, marker="o", label="Treino")
plt.plot(profundidades, acc_val, marker="o", label="Teste")
plt.xlabel("max_depth")
plt.ylabel("Acurácia")
plt.title("Acurácia x profundidade da árvore")
plt.legend()
plt.show()

## 8. Avaliação do melhor modelo no conjunto de teste

In [ ]:
melhor_arvore = grid.best_estimator_

res_melhor = avaliar(melhor_arvore, X_train, y_train, X_test, y_test, "Árvore ajustada")
pd.concat([res_base, res_melhor], axis=1).round(4)

In [ ]:
y_pred = melhor_arvore.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Rejected (0)", "Approved (1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_estimator(
    melhor_arvore, X_test, y_test,
    display_labels=["Rejected", "Approved"], cmap="Blues", ax=axes[0],
)
axes[0].set_title("Matriz de confusão")

RocCurveDisplay.from_estimator(arvore_base, X_test, y_test, name="Árvore base", ax=axes[1])
RocCurveDisplay.from_estimator(melhor_arvore, X_test, y_test, name="Árvore ajustada", ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", label="Aleatório")
axes[1].set_title("Curva ROC")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Interpretação do modelo

### 9.1 Importância das variáveis

In [ ]:
importancias = pd.Series(melhor_arvore.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(9, 5))
importancias.plot.barh(color="steelblue")
plt.title("Importância das variáveis (redução de impureza)")
plt.xlabel("Importância")
plt.show()

importancias.sort_values(ascending=False).round(4)

### 9.2 Visualização da árvore

In [ ]:
# Exibe no máximo os 4 primeiros níveis para manter a figura legível
plt.figure(figsize=(24, 10))
plot_tree(
    melhor_arvore,
    feature_names=X.columns,
    class_names=["Rejected", "Approved"],
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=9,
)
plt.title("Árvore de decisão ajustada (até 4 níveis)")
plt.show()

### 9.3 Regras de decisão em texto

In [ ]:
print(export_text(melhor_arvore, feature_names=list(X.columns), max_depth=4))

## 10. Exemplo de previsão para um novo cliente

In [ ]:
novo_cliente = pd.DataFrame([{
    "no_of_dependents": 2,
    "education": 1,              # Graduate
    "self_employed": 0,          # No
    "income_annum": 6_000_000,
    "loan_amount": 15_000_000,
    "loan_term": 10,
    "cibil_score": 720,
    "residential_assets_value": 5_000_000,
    "commercial_assets_value": 3_000_000,
    "luxury_assets_value": 10_000_000,
    "bank_asset_value": 4_000_000,
}])[X.columns]

pred = melhor_arvore.predict(novo_cliente)[0]
proba = melhor_arvore.predict_proba(novo_cliente)[0, 1]
print("Previsão:", "Approved" if pred == 1 else "Rejected")
print("Probabilidade de aprovação: %.2f" % proba)

## 11. Conclusões

- O problema é de **classificação binária** (`Approved` x `Rejected`), com classes moderadamente desbalanceadas (~62% aprovados).
- A **árvore base** atinge 100% de acurácia no treino, sinal de *overfitting*; o ajuste de hiperparâmetros produz uma árvore **mais rasa e mais simples**, com desempenho no teste igual ou superior.
- O **`cibil_score`** (score de crédito) tende a ser, de longe, a variável mais importante, dominando as primeiras divisões da árvore — confira o gráfico da seção 9.1 para a ordem das demais variáveis.
- Uma vantagem da árvore de decisão é a **interpretabilidade**: as regras extraídas (seção 9.3) podem ser lidas diretamente como critérios de concessão de crédito.